# 📐 NB 0b – Annotierte Bilder skalieren
**Dataset/images + Dataset/labels → verkleinert, Annotationen unverändert**

| Abschnitt | Inhalt |
|-----------|--------|
| 1 | Konfiguration |
| 2 | Hilfsfunktionen |
| 3 | Schreibschutz prüfen |
| 4 | Bilder skalieren (Annotationen bleiben unverändert) |
| 5 | Verifikation – Annotationen vor/nach vergleichen |
| 6 | Visuelle Kontrolle mit Bounding Boxes |

> 💡 **Warum bleiben Annotationen unverändert?**
> YOLO speichert Koordinaten **normalisiert** (0.0–1.0 relativ zur Bildgröße).
> Ein Punkt bei 30% von links bleibt bei 30% von links — egal ob das Bild
> 4000px oder 640px breit ist. Kein Umrechnen nötig.

---

---
## ⚙️ Abschnitt 1 – Konfiguration

```
../
└── Dataset/                 ← Eingabe (original annotiert)
    ├── images/
    │   ├── 20260402_xxx.jpg
    │   └── ...
    ├── labels/
    │   ├── 20260402_xxx.txt
    │   └── ...
    └── classes.txt

└── Dataset_skaliert/        ← Ausgabe (neu erstellt)
    ├── images/              (skalierte Bilder)
    ├── labels/              (KOPIE – unverändert)
    └── classes.txt          (KOPIE)
```

In [1]:
import cv2, shutil, pathlib, os, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.image as mpimg
from IPython.display import display

# ╔═══════════════════════════════════════════════╗
# ║  KONFIGURATION – hier anpassen                       ║
# ╚═══════════════════════════════════════════════╝

# Eingabe: annotierter Datensatz
DATASET_DIR  = pathlib.Path(r'../Dataset')

# Ausgabe: skalierter Datensatz (wird neu erstellt)
OUTPUT_DIR   = pathlib.Path(r'../Dataset_skaliert')

# Zielgröße in Pixel (längste Seite)
# Typische YOLO-Trainingsauflösungen: 320, 416, 512, 640, 800, 1280
ZIELGROESSE  = 1280     # px – längste Seite des Bildes

# Skalierungsmethode
# 'letterbox' : Seitenverhältnis beibehalten, Rest schwarz auffüllen
#               → alle Bilder exakt ZIELGROESSE x ZIELGROESSE
# 'fit'       : Seitenverhältnis beibehalten, KEINE schwarzen Balken
#               → längste Seite = ZIELGROESSE, andere Seite kleiner
METHODE      = 'fit'   # 'letterbox' oder 'fit'

JPEG_QUALITAET = 92
IMG_EXT        = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# ── Pfade ableiten ───────────────────────────────────
SRC_IMAGES   = DATASET_DIR / 'images'
SRC_LABELS   = DATASET_DIR / 'labels'
DST_IMAGES   = OUTPUT_DIR  / 'images'
DST_LABELS   = OUTPUT_DIR  / 'labels'

print('=' * 55)
print(f'  Eingabe     : {DATASET_DIR}')
print(f'  Ausgabe     : {OUTPUT_DIR}')
print(f'  Zielgröße   : {ZIELGROESSE} px (längste Seite)')
print(f'  Methode     : {METHODE}')
print(f'  JPEG-Qual.  : {JPEG_QUALITAET}')
print('=' * 55)
print()
print('💡 Annotationen (YOLO .txt) werden NICHT verändert –')
print('   normalisierte Koordinaten bleiben bei jeder Auflösung korrekt.')


  Eingabe     : ..\Dataset
  Ausgabe     : ..\Dataset_skaliert
  Zielgröße   : 1280 px (längste Seite)
  Methode     : fit
  JPEG-Qual.  : 92

💡 Annotationen (YOLO .txt) werden NICHT verändert –
   normalisierte Koordinaten bleiben bei jeder Auflösung korrekt.


---
## 🔧 Abschnitt 2 – Hilfsfunktionen

In [2]:
def skaliere_bild(img, zielgroesse, methode='fit'):
    """
    Skaliert ein Bild auf die Zielgröße.
    methode='fit'       : längste Seite = zielgroesse, kein Beschnitt
    methode='letterbox' : quadratisch mit schwarzen Balken
    Gibt (skaliertes Bild, scale_factor) zurück.
    """
    h, w   = img.shape[:2]
    scale  = zielgroesse / max(w, h)
    nw, nh = int(w * scale), int(h * scale)
    resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)

    if methode == 'letterbox':
        canvas = np.zeros((zielgroesse, zielgroesse, 3), dtype=np.uint8)
        x0 = (zielgroesse - nw) // 2
        y0 = (zielgroesse - nh) // 2
        canvas[y0:y0+nh, x0:x0+nw] = resized
        return canvas, scale
    else:  # fit
        return resized, scale

def lese_label(lbl_path):
    """Liest YOLO-Label und gibt Liste von (cls, xc, yc, bw, bh) zurück."""
    if not lbl_path.exists():
        return []
    boxen = []
    for zeile in lbl_path.read_text(encoding='utf-8').splitlines():
        t = zeile.strip().split()
        if len(t) == 5:
            boxen.append((int(t[0]), *map(float, t[1:])))
    return boxen

def zeichne_boxen(ax, img_rgb, boxen, class_names, titel):
    """Zeichnet Bounding Boxes auf eine Matplotlib-Achse."""
    FARBEN = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6','#1abc9c']
    H, W   = img_rgb.shape[:2]
    ax.imshow(img_rgb)
    ax.axis('off')
    ax.set_title(titel, fontsize=9, pad=4)
    for cid, xc, yc, bw, bh in boxen:
        farbe = FARBEN[cid % len(FARBEN)]
        name  = class_names[cid] if cid < len(class_names) else f'cls_{cid}'
        x1 = (xc - bw/2) * W
        y1 = (yc - bh/2) * H
        ax.add_patch(patches.Rectangle(
            (x1, y1), bw*W, bh*H,
            linewidth=2, edgecolor=farbe, facecolor=farbe+'33'))
        ax.text(x1+3, y1-5, name, color='white', fontsize=8,
                fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', fc=farbe,
                          alpha=0.85, ec='none'))

print('✅ Hilfsfunktionen geladen')


✅ Hilfsfunktionen geladen


---
## 🔒 Abschnitt 3 – Schreibschutz prüfen

Prüft ob alle Quelldateien lesbar sind (Schreibschutz spielt hier keine Rolle,
da die Originale **nicht verändert** werden — alles landet in `Dataset_skaliert/`).

> ✅ Dieser Check stellt sicher dass alle Bilder gelesen werden können.

In [3]:
bilder = sorted([p for p in SRC_IMAGES.iterdir()
                 if p.suffix.lower() in IMG_EXT])

if not bilder:
    print(f'⚠️  Keine Bilder in {SRC_IMAGES}')
else:
    nicht_lesbar = [b for b in bilder if not os.access(str(b), os.R_OK)]
    kein_label   = [b for b in bilder
                    if not (SRC_LABELS / (b.stem + '.txt')).exists()]

    print(f'🔍 Prüfe {len(bilder)} Bilder in {SRC_IMAGES}\n' + '-'*55)
    print(f'  Bilder gesamt        : {len(bilder)}')
    print(f'  Nicht lesbar         : {len(nicht_lesbar)}')
    print(f'  Ohne Label (.txt)    : {len(kein_label)}')

    if nicht_lesbar:
        print('\n❌ Nicht lesbare Dateien:')
        for f in nicht_lesbar:
            print(f'   {f.name}')

    if kein_label:
        print('\n⚠️  Bilder ohne Label (werden trotzdem skaliert):')
        for f in kein_label[:5]:
            print(f'   {f.name}')
        if len(kein_label) > 5:
            print(f'   ... ({len(kein_label)-5} weitere)')

    print('\n' + '='*55)
    if not nicht_lesbar:
        print('✅ Alle Bilder lesbar – bereit für Schritt 4.')
        # Originalgrößen stichprobenartig zeigen
        probe = bilder[:3]
        print('\nStichprobe Originalauflösungen:')
        for b in probe:
            img = cv2.imread(str(b))
            if img is not None:
                h, w = img.shape[:2]
                print(f'  {b.name:<35} {w}×{h} px')
    else:
        print('⚠️  Bitte nicht lesbare Dateien beheben.')


🔍 Prüfe 65 Bilder in ..\Dataset\images
-------------------------------------------------------
  Bilder gesamt        : 65
  Nicht lesbar         : 0
  Ohne Label (.txt)    : 0

✅ Alle Bilder lesbar – bereit für Schritt 4.

Stichprobe Originalauflösungen:
  20260402_095045.jpg                 4000×3000 px
  20260402_095048.jpg                 4000×3000 px
  20260402_095052.jpg                 4000×3000 px


---
## 📐 Abschnitt 4 – Bilder skalieren

Skaliert alle Bilder aus `Dataset/images/` und kopiert die Labels **unverändert**.
`Dataset_skaliert/` wird komplett neu erstellt.

**Warum bleiben Annotationen unverändert?**

```
Original: 4000×3000 px
LED-Box:  xc=0.45  yc=0.32  bw=0.08  bh=0.12
          → Pixel: x=1800, y=960, w=320, h=360

Skaliert: 640×480 px  (Faktor 0.16)
LED-Box:  xc=0.45  yc=0.32  bw=0.08  bh=0.12  ← IDENTISCH
          → Pixel: x=288, y=154, w=51, h=58
```
Die normalisierten Werte bleiben exakt gleich — YOLO rechnet intern um.

In [4]:
# Ausgabeordner neu erstellen
for ordner in (DST_IMAGES, DST_LABELS):
    if ordner.exists():
        shutil.rmtree(ordner)
    ordner.mkdir(parents=True)

# classes.txt kopieren
classes_src = DATASET_DIR / 'classes.txt'
if classes_src.exists():
    shutil.copy2(classes_src, OUTPUT_DIR / 'classes.txt')

bilder = sorted([p for p in SRC_IMAGES.iterdir()
                 if p.suffix.lower() in IMG_EXT])

ok = kein_label = fehler = 0
groessen_vorher  = []
groessen_nachher = []

print(f'Skaliere {len(bilder)} Bilder  '
      f'(Methode: {METHODE}, Ziel: {ZIELGROESSE}px)\n' + '-'*62)

for b in bilder:
    img = cv2.imread(str(b))
    if img is None:
        print(f'  ⚠️  {b.name} – konnte nicht geladen werden')
        fehler += 1
        continue

    h_vor, w_vor = img.shape[:2]
    groessen_vorher.append((w_vor, h_vor))

    # Bild skalieren
    img_out, scale = skaliere_bild(img, ZIELGROESSE, METHODE)
    h_nach, w_nach = img_out.shape[:2]
    groessen_nachher.append((w_nach, h_nach))

    # Bild speichern
    ziel_bild = DST_IMAGES / b.with_suffix('.jpg').name
    cv2.imwrite(str(ziel_bild), img_out,
                [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITAET])

    # Label UNVERÄNDERT kopieren
    src_lbl = SRC_LABELS / (b.stem + '.txt')
    dst_lbl = DST_LABELS / (b.stem + '.txt')
    if src_lbl.exists():
        shutil.copy2(src_lbl, dst_lbl)
        lbl_info = '✓ Label'
        ok += 1
    else:
        lbl_info = '⚠️ kein Label'
        kein_label += 1

    print(f'  ✅  {b.name:<35}  '
          f'{w_vor}×{h_vor} → {w_nach}×{h_nach}  {lbl_info}')

# Zusammenfassung
print('-'*62)
print(f'✅ Skaliert: {ok+kein_label}  |  '
      f'✓ Mit Label: {ok}  |  ⚠️ Ohne Label: {kein_label}  |  '
      f'❌ Fehler: {fehler}')
if groessen_vorher:
    avg_w_vor = sum(w for w,h in groessen_vorher)  / len(groessen_vorher)
    avg_h_vor = sum(h for w,h in groessen_vorher)  / len(groessen_vorher)
    avg_w_nch = sum(w for w,h in groessen_nachher) / len(groessen_nachher)
    avg_h_nch = sum(h for w,h in groessen_nachher) / len(groessen_nachher)
    red = (1 - (avg_w_nch * avg_h_nch) / (avg_w_vor * avg_h_vor)) * 100
    print(f'\nDurchschnitt vorher : {avg_w_vor:.0f}×{avg_h_vor:.0f} px')
    print(f'Durchschnitt nachher: {avg_w_nch:.0f}×{avg_h_nch:.0f} px')
    print(f'Dateimengen-Reduktion: ∼{red:.0f}% kleinere Bilddaten')
print(f'\n→ Ausgabe: {OUTPUT_DIR.resolve()}')


Skaliere 65 Bilder  (Methode: fit, Ziel: 1280px)
--------------------------------------------------------------
  ✅  20260402_095045.jpg                  4000×3000 → 1280×960  ✓ Label
  ✅  20260402_095048.jpg                  4000×3000 → 1280×960  ✓ Label
  ✅  20260402_095052.jpg                  4000×3000 → 1280×960  ✓ Label
  ✅  20260402_095057.jpg                  4000×3000 → 1280×960  ✓ Label
  ✅  20260402_095059.jpg                  4000×3000 → 1280×960  ✓ Label
  ✅  20260402_095113.jpg                  4000×3000 → 1280×960  ✓ Label
  ✅  20260402_095117.jpg                  4000×3000 → 1280×960  ✓ Label
  ✅  20260402_095120.jpg                  4000×3000 → 1280×960  ✓ Label
  ✅  20260402_095127.jpg                  4000×3000 → 1280×960  ✓ Label
  ✅  20260402_095132.jpg                  4000×3000 → 1280×960  ✓ Label
  ✅  20260402_095137.jpg                  4000×3000 → 1280×960  ✓ Label
  ✅  20260402_095141.jpg                  4000×3000 → 1280×960  ✓ Label
  ✅  20260402_095146.jpg

---
## 🔬 Abschnitt 5 – Verifikation: Annotationen unverändert?

Dieser Check beweist dass die Label-Dateien **byte-identisch** kopiert wurden:
- Vergleicht Original-`.txt` mit kopierter `.txt` Zeile für Zeile
- Meldet sofort wenn eine Abweichung gefunden wird
- Zeigt Stichproben der Koordinatenwerte

> ✅ Erwartetes Ergebnis: 0 Abweichungen — alle Werte identisch.

In [5]:
bilder = sorted([p for p in DST_IMAGES.iterdir()
                 if p.suffix.lower() in IMG_EXT])

abweichungen = 0
geprueft     = 0
probe_zeilen = []  # Stichproben zum Anzeigen

print(f'🔍 Vergleiche Labels: Original ↔ Skaliert\n' + '-'*58)

for b in bilder:
    src_lbl = SRC_LABELS  / (b.stem + '.txt')
    dst_lbl = DST_LABELS  / (b.stem + '.txt')

    if not src_lbl.exists() or not dst_lbl.exists():
        continue

    src_text = src_lbl.read_text(encoding='utf-8').strip()
    dst_text = dst_lbl.read_text(encoding='utf-8').strip()

    if src_text != dst_text:
        abweichungen += 1
        print(f'  ❌ ABWEICHUNG: {b.stem}')
        print(f'     Original : {src_text[:80]}')
        print(f'     Kopie    : {dst_text[:80]}')
    else:
        geprueft += 1
        # Erste 3 als Stichprobe merken
        if len(probe_zeilen) < 3 and src_text:
            probe_zeilen.append((b.stem, src_text.splitlines()[0]))

print(f'✅ Geprüft     : {geprueft} Label-Dateien')
print(f'❌ Abweichungen: {abweichungen}')
print()

if abweichungen == 0:
    print('✅✅ BESTÄTIGT: Alle Annotationen sind identisch!')
    print('   Die normalisierten YOLO-Koordinaten passen zu jeder Auflösung.')
else:
    print('⚠️  WARNUNG: Abweichungen gefunden! Bitte oben prüfen.')

# Stichproben anzeigen
if probe_zeilen:
    print('\nStichprobe (erste Zeile je Label) – Original = Kopie:')
    print('-'*58)
    for stem, zeile in probe_zeilen:
        t = zeile.split()
        print(f'  {stem[:30]}')
        print(f'    cls={t[0]}  xc={t[1]}  yc={t[2]}  bw={t[3]}  bh={t[4]}')
        print(f'    → Diese Werte gelten für Original UND skaliertes Bild ✓')

# Größenvergleich
print('\nGrößenvergleich (3 Beispiele):')
print('-'*58)
for b in list(DST_IMAGES.iterdir())[:3]:
    if b.suffix.lower() not in IMG_EXT: continue
    src_b = SRC_IMAGES / b.name
    img_src = cv2.imread(str(src_b))
    img_dst = cv2.imread(str(b))
    if img_src is not None and img_dst is not None:
        hs, ws = img_src.shape[:2]
        hd, wd = img_dst.shape[:2]
        faktor = wd / ws
        print(f'  {b.name[:35]}')
        print(f'    Original : {ws}×{hs} px')
        print(f'    Skaliert : {wd}×{hd} px  (Faktor {faktor:.3f})')
        print(f'    Annotation: unverändert ✓')


🔍 Vergleiche Labels: Original ↔ Skaliert
----------------------------------------------------------
✅ Geprüft     : 65 Label-Dateien
❌ Abweichungen: 0

✅✅ BESTÄTIGT: Alle Annotationen sind identisch!
   Die normalisierten YOLO-Koordinaten passen zu jeder Auflösung.

Stichprobe (erste Zeile je Label) – Original = Kopie:
----------------------------------------------------------
  20260402_095045
    cls=0  xc=0.692923  yc=0.504944  bw=0.037852  bh=0.201875
    → Diese Werte gelten für Original UND skaliertes Bild ✓
  20260402_095048
    cls=0  xc=0.869147  yc=0.819477  bw=0.097905  bh=0.193140
    → Diese Werte gelten für Original UND skaliertes Bild ✓
  20260402_095052
    cls=0  xc=0.308968  yc=0.111893  bw=0.121025  bh=0.091566
    → Diese Werte gelten für Original UND skaliertes Bild ✓

Größenvergleich (3 Beispiele):
----------------------------------------------------------
  20260402_095045.jpg
    Original : 4000×3000 px
    Skaliert : 1280×960 px  (Faktor 0.320)
    Annotation: 

---
## 🔍 Abschnitt 6 – Visuelle Kontrolle

Zeigt Original und skaliertes Bild nebeneinander **mit Bounding Boxes**.
Die Boxen sollten in beiden Bildern an identischer relativer Position sitzen.

> ✅ Wenn die Boxen in Original und skaliertem Bild die gleichen Objekte
> umschließen, ist die Skalierung korrekt.

In [ ]:
%matplotlib inline
plt.close('all')

# ── Einstellungen ─────────────────────────────────
ANZAHL_KONTROLLE = 3   # Wie viele Bildpaare anzeigen
MODUS_K = 'zufall'     # 'zufall' oder 'bereich'
VON_K   = 1
BIS_K   = 5

# Klassennamen laden
classes_file = OUTPUT_DIR / 'classes.txt'
CLASS_NAMES  = ([l.strip() for l in classes_file.read_text().splitlines() if l.strip()]
                if classes_file.exists() else [])
print(f'Klassen: {CLASS_NAMES}')

# Bilder auswählen
alle = sorted([p for p in DST_IMAGES.iterdir()
               if p.suffix.lower() in IMG_EXT
               and (DST_LABELS / (p.stem + '.txt')).exists()])

if not alle:
    print('⚠️  Keine Bilder mit Labels gefunden.')
else:
    if MODUS_K == 'zufall':
        auswahl = random.sample(alle, min(ANZAHL_KONTROLLE, len(alle)))
    else:
        auswahl = alle[VON_K-1:BIS_K]

    fig, axes = plt.subplots(len(auswahl), 2,
                             figsize=(14, 5 * len(auswahl)))
    if len(auswahl) == 1:
        axes = [axes]

    for row, dst_p in enumerate(auswahl):
        src_p   = SRC_IMAGES / dst_p.name
        lbl_p   = DST_LABELS / (dst_p.stem + '.txt')
        boxen   = lese_label(lbl_p)

        # Original laden
        img_src = cv2.imread(str(src_p))
        img_dst = cv2.imread(str(dst_p))

        if img_src is None or img_dst is None:
            continue

        hs, ws = img_src.shape[:2]
        hd, wd = img_dst.shape[:2]

        zeichne_boxen(
            axes[row][0],
            cv2.cvtColor(img_src, cv2.COLOR_BGR2RGB),
            boxen, CLASS_NAMES,
            f'ORIGINAL: {dst_p.name}\n{ws}×{hs} px'
        )
        zeichne_boxen(
            axes[row][1],
            cv2.cvtColor(img_dst, cv2.COLOR_BGR2RGB),
            boxen, CLASS_NAMES,
            f'SKALIERT: {dst_p.name}\n{wd}×{hd} px  '
            f'(Faktor {wd/ws:.2f})'
        )

    plt.suptitle(
        f'Vergleich: Original ↔ Skaliert ({ZIELGROESSE}px, Methode: {METHODE})\n'
        f'Gleiche Label-Datei für beide Bilder – Boxen sollen übereinstimmen',
        fontsize=11, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    display(fig)
    plt.close()
    print(f'✅ {len(auswahl)} Bildpaare angezeigt')
